<a href="https://colab.research.google.com/github/ga426553-sudo/Classification-of-Breast-Cancer-Subtypes-machine-learning---CuMiDa-22820/blob/main/RandomForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Séptimo código - Cáncer de Mama 🌸

**Implementación de Random Forest para Clasificación de Cáncer de Mama**

###Importar librerías necesarias 🌺

In [ ]:
# ============================================
# CÓDIGO 7: RANDOM FOREST Y EVALUACIÓN
# ============================================

from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

print("="*60)
print("🌲 CÓDIGO 7: EVALUACIÓN DE RANDOM FOREST")
print("="*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🌲 CÓDIGO 7: EVALUACIÓN DE RANDOM FOREST


###Cargar datos 🌺

In [ ]:
# 1. CARGAR DATOS
# ================
print(f"\n📂 Cargando datos del Código 2...")

with open('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/subsets.pkl', 'rb') as f:
    subsets = pickle.load(f)

y_train = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/y_train_final.npy')
y_test = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/y_test_final.npy')

print(f"✅ Datos cargados:")
print(f"   Subconjuntos: {list(subsets.keys())}")
print(f"   y_train: {y_train.shape}")
print(f"   y_test: {y_test.shape}")


📂 Cargando datos del Código 2...
✅ Datos cargados:
   Subconjuntos: ['EO', 'BBA', 'CSA', 'RDA', 'GA']
   y_train: (206,)
   y_test: (28,)


###Configuración de random forest 🌺

In [ ]:
# 2. CONFIGURAR RANDOM FOREST (basado en el artículo, similar a XGBoost)
# ========================================================================
rf_params = {
    'n_estimators': 100,
    'max_depth': 4,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'criterion': 'gini',
    'random_state': 42
}

rf_model = RandomForestClassifier(**rf_params)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

print(f"\n⚙️ Configuración de Random Forest:")
for key, value in rf_params.items():
    print(f"   {key}: {value}")


⚙️ Configuración de Random Forest:
   n_estimators: 100
   max_depth: 4
   min_samples_split: 2
   min_samples_leaf: 1
   criterion: gini
   random_state: 42


### Evaluación de Random Forest 🌺

In [ ]:
# 3. EVALUAR CADA SUBCONJUNTO
# ============================
results = {}

for name in subsets.keys():
    print(f"\n{'─'*40}")
    print(f"🔍 Subconjunto: {name} - Genes: {subsets[name]}")

    X_train_subset = np.load(f'/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/train_{name}.npy')
    X_test_subset = np.load(f'/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/test_{name}.npy')

    # Validación cruzada (10-fold)
    cv_acc = cross_val_score(rf_model, X_train_subset, y_train, cv=cv, scoring='accuracy')
    cv_f1 = cross_val_score(rf_model, X_train_subset, y_train, cv=cv, scoring='f1')
    cv_auc = cross_val_score(rf_model, X_train_subset, y_train, cv=cv, scoring='roc_auc')

    # Entrenar modelo completo
    rf_model.fit(X_train_subset, y_train)

    # Predicciones
    y_pred = rf_model.predict(X_test_subset)
    y_proba = rf_model.predict_proba(X_test_subset)[:, 1]

    # Métricas en test
    test_acc = accuracy_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred)
    test_auc = roc_auc_score(y_test, y_proba)

    # Importancia de características
    feature_importance = pd.DataFrame({
        'gene': subsets[name],
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)

    results[name] = {
        'cv_accuracy': f"{cv_acc.mean():.4f} ± {cv_acc.std():.4f}",
        'cv_f1': f"{cv_f1.mean():.4f} ± {cv_f1.std():.4f}",
        'cv_auc': f"{cv_auc.mean():.4f} ± {cv_auc.std():.4f}",
        'test_accuracy': test_acc,
        'test_f1': test_f1,
        'test_auc': test_auc,
        'genes': subsets[name],
        'feature_importance': feature_importance,
        'y_pred': y_pred,
        'y_proba': y_proba
    }

    print(f"\n   Resultados:")
    print(f"   📊 Validación Cruzada (10-fold):")
    print(f"      Accuracy: {results[name]['cv_accuracy']}")
    print(f"      F1-Score: {results[name]['cv_f1']}")
    print(f"      AUC: {results[name]['cv_auc']}")
    print(f"\n   📈 Test:")
    print(f"      Accuracy: {test_acc:.4f}")
    print(f"      F1-Score: {test_f1:.4f}")
    print(f"      AUC: {test_auc:.4f}")
    print(f"\n   🔬 Importancia de genes:")
    print(feature_importance.to_string(index=False))


────────────────────────────────────────
🔍 Subconjunto: EO - Genes: ['NM_002644', 'BM511013', 'NM_182611']

   Resultados:
   📊 Validación Cruzada (10-fold):
      Accuracy: 0.9952 ± 0.0143
      F1-Score: 0.9957 ± 0.0130
      AUC: 1.0000 ± 0.0000

   📈 Test:
      Accuracy: 1.0000
      F1-Score: 1.0000
      AUC: 1.0000

   🔬 Importancia de genes:
     gene  importance
 BM511013    0.364107
NM_002644    0.356963
NM_182611    0.278930

────────────────────────────────────────
🔍 Subconjunto: BBA - Genes: ['NM_002644', 'BM511013']

   Resultados:
   📊 Validación Cruzada (10-fold):
      Accuracy: 0.9952 ± 0.0143
      F1-Score: 0.9947 ± 0.0158
      AUC: 1.0000 ± 0.0000

   📈 Test:
      Accuracy: 0.9643
      F1-Score: 0.9811
      AUC: 1.0000

   🔬 Importancia de genes:
     gene  importance
 BM511013    0.515424
NM_002644    0.484576

────────────────────────────────────────
🔍 Subconjunto: CSA - Genes: ['BM511013', 'NM_182611']

   Resultados:
   📊 Validación Cruzada (10-fold):
   

###Resultados 🌺

In [ ]:
# 4. MOSTRAR RESULTADOS COMPARATIVOS
# ===================================
print(f"\n{'='*60}")
print("📊 RESULTADOS FINALES - RANDOM FOREST")
print('='*60)

comparison_df = pd.DataFrame({
    'Subset': results.keys(),
    'Genes': [', '.join(r['genes']) for r in results.values()],
    'Test_Accuracy': [f"{r['test_accuracy']:.4f}" for r in results.values()],
    'Test_F1': [f"{r['test_f1']:.4f}" for r in results.values()],
    'Test_AUC': [f"{r['test_auc']:.4f}" for r in results.values()],
    'CV_Accuracy': [r['cv_accuracy'] for r in results.values()]
})

print("\n📋 Tabla comparativa:")
print(comparison_df.to_string(index=False))


📊 RESULTADOS FINALES - RANDOM FOREST

📋 Tabla comparativa:
Subset                          Genes Test_Accuracy Test_F1 Test_AUC     CV_Accuracy
    EO NM_002644, BM511013, NM_182611        1.0000  1.0000   1.0000 0.9952 ± 0.0143
   BBA            NM_002644, BM511013        0.9643  0.9811   1.0000 0.9952 ± 0.0143
   CSA            BM511013, NM_182611        1.0000  1.0000   1.0000 0.9952 ± 0.0143
   RDA                      NM_002644        0.9286  0.9630   0.6827 0.9807 ± 0.0236
    GA                       AW015426        1.0000  1.0000   1.0000 0.9950 ± 0.0150
